In [1]:
"""
Interactive 3D Optimization with Plotly
Rotate, zoom, and explore the parameter space!
"""

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime

print("✅ Imports successful!")
print("🚀 Starting optimization...\n")

# Download BTC data to present day
ticker = 'BTC-USD'
start_date = '2020-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')

print(f"Downloading {ticker} from {start_date} to {end_date}...")
df = yf.download(ticker, start=start_date, end=end_date, progress=False)

print(f"✅ Downloaded {len(df)} days of data")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}\n")

✅ Imports successful!
🚀 Starting optimization...

✅ Downloaded 2225 days of data
Date range: 2020-01-01 to 2026-02-02



In [2]:
# Backtest function
def backtest_sma(df, short_window, long_window):
    """Backtest SMA strategy and return metrics"""
    df = df.copy()
    
    # Calculate SMAs
    df['SMA_Short'] = df['Close'].rolling(short_window).mean()
    df['SMA_Long'] = df['Close'].rolling(long_window).mean()
    
    # Generate signals
    df['Signal'] = np.where(df['SMA_Short'] > df['SMA_Long'], 1, 0)
    df['Position'] = df['Signal'].shift(1)
    
    # Calculate returns
    df['Market_Returns'] = df['Close'].pct_change()
    df['Strategy_Returns'] = df['Position'] * df['Market_Returns']
    
    # Drop NaN
    df = df.dropna()
    
    if len(df) == 0 or df['Strategy_Returns'].std() == 0:
        return 0, 0, 0
    
    # Sharpe Ratio
    mean_return = df['Strategy_Returns'].mean()
    std_return = df['Strategy_Returns'].std()
    sharpe = (mean_return / std_return) * np.sqrt(252)
    
    # Total Return
    total_return = (1 + df['Strategy_Returns']).prod() - 1
    
    # Max Drawdown
    cumulative = (1 + df['Strategy_Returns']).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    max_dd = abs(drawdown.min())
    
    return sharpe, total_return, max_dd

# Run optimization
short_range = range(4, 50, 2)
long_range = range(10, 300, 3)

print(f"Testing {len(short_range)} × {len(long_range)} parameter combinations...")

results = []
counter = 0
total_combos = len(short_range) * len(long_range)

for short in short_range:
    for long in long_range:
        if short >= long:
            continue
        
        counter += 1
        if counter % 10 == 0:
            print(f"Progress: {counter}/{total_combos}")
        
        sharpe, total_return, max_dd = backtest_sma(df, short, long)
        
        results.append({
            'short': short,
            'long': long,
            'sharpe': sharpe,
            'return': total_return * 100,
            'max_dd': max_dd * 100
        })

results_df = pd.DataFrame(results)

print(f"\n✅ Optimization complete! Tested {len(results_df)} combinations")

# Find best
best = results_df.loc[results_df['sharpe'].idxmax()]
print(f"\nBest Parameters Found:")
print(f"  Short SMA: {best['short']:.0f}")
print(f"  Long SMA:  {best['long']:.0f}")
print(f"  Sharpe:    {best['sharpe']:.2f}")
print(f"  Return:    {best['return']:.2f}%")
print(f"  Max DD:    {best['max_dd']:.2f}%")


Testing 23 × 97 parameter combinations...
Progress: 10/2231
Progress: 20/2231
Progress: 30/2231
Progress: 40/2231
Progress: 50/2231
Progress: 60/2231
Progress: 70/2231
Progress: 80/2231
Progress: 90/2231
Progress: 100/2231
Progress: 110/2231
Progress: 120/2231
Progress: 130/2231
Progress: 140/2231
Progress: 150/2231
Progress: 160/2231
Progress: 170/2231
Progress: 180/2231
Progress: 190/2231
Progress: 200/2231
Progress: 210/2231
Progress: 220/2231
Progress: 230/2231
Progress: 240/2231
Progress: 250/2231
Progress: 260/2231
Progress: 270/2231
Progress: 280/2231
Progress: 290/2231
Progress: 300/2231
Progress: 310/2231
Progress: 320/2231
Progress: 330/2231
Progress: 340/2231
Progress: 350/2231
Progress: 360/2231
Progress: 370/2231
Progress: 380/2231
Progress: 390/2231
Progress: 400/2231
Progress: 410/2231
Progress: 420/2231
Progress: 430/2231
Progress: 440/2231
Progress: 450/2231
Progress: 460/2231
Progress: 470/2231
Progress: 480/2231
Progress: 490/2231
Progress: 500/2231
Progress: 510/223

In [3]:
# Create interactive 3D surface: Returns (height) colored by Sharpe
fig = go.Figure(data=[go.Surface(
    x=X,
    y=Y,
    z=results_df.pivot(index='long', columns='short', values='return').values,
    surfacecolor=Z,  # Color by Sharpe
    colorscale='RdYlGn',
    colorbar=dict(title="Sharpe<br>Ratio", len=0.7),
    hovertemplate='Short: %{x}<br>Long: %{y}<br>Return: %{z:.1f}%<br>Sharpe: %{surfacecolor:.3f}<extra></extra>'
)])

# Add red diamond for best point
fig.add_trace(go.Scatter3d(
    x=[best['short']],
    y=[best['long']],
    z=[best['return']],
    mode='markers',
    marker=dict(size=12, color='red', symbol='diamond'),
    name=f"Best ({best['short']:.0f}/{best['long']:.0f})",
    hovertemplate='<b>BEST</b><br>Short: %{x}<br>Long: %{y}<br>Return: %{z:.1f}%<extra></extra>'
))

fig.update_layout(
    title=f'Interactive 3D: Returns (Height) colored by Sharpe Ratio<br>{ticker} ({start_date} to {end_date})',
    scene=dict(
        xaxis_title='Short SMA Window (days)',
        yaxis_title='Long SMA Window (days)',
        zaxis_title='Total Return (%)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.3))
    ),
    width=1000,
    height=800,
    showlegend=True
)

print("✅ Returns vs Sharpe surface ready!")
print("📈 Height = Return %")
print("🎨 Color = Sharpe Ratio (Green=Best, Red=Worst)")
print("🎮 Rotate to explore!\n")

fig.show()

NameError: name 'X' is not defined